In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/icetraythegag/competition-data/test_transactions.csv
/kaggle/input/datasets/icetraythegag/competition-data/y_train.csv
/kaggle/input/datasets/icetraythegag/competition-data/train_transactions.csv


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'autogluon.tabular', '-q'], check=True)

import pandas as pd, numpy as np
from autogluon.tabular import TabularPredictor
import warnings; warnings.filterwarnings('ignore')

FEAT_DIR = '/kaggle/input/datasets/icetraythegag/competition-data'  # ← dataset 이름 맞춰줘
OUT_DIR  = '/kaggle/working'

# === 데이터 로드 ===
X = pd.read_parquet(f'{FEAT_DIR}/feats_attack200.parquet')
if 'custid' in X.columns: X = X.set_index('custid')
y_train = pd.read_csv(f'{FEAT_DIR}/y_train.csv')
tr_ids = y_train['custid'].values; tr_set = set(tr_ids)
test_ids = [c for c in X.index if c not in tr_set]
t = y_train.set_index('custid')['gender'].loc[tr_ids].values
X = X.fillna(0)
print(f'피쳐: {X.shape[1]}개 | train: {len(tr_ids)} | test: {len(test_ids)}')

# === AutoGluon 4시간 ===
train_data = X.loc[tr_ids].copy(); train_data['gender'] = t
X_test = X.loc[test_ids]

predictor = TabularPredictor(
    label='gender',
    problem_type='binary',
    eval_metric='roc_auc',
    path=f'{OUT_DIR}/ag_model'
).fit(
    train_data,
    presets='best_quality',
    time_limit=14400,
    dynamic_stacking=False,
    num_stack_levels=0
)

pred = predictor.predict_proba(X_test)[1]
pd.DataFrame({'custid': X_test.index, 'gender': pred}).to_csv(
    f'{OUT_DIR}/submission_attack_4h.csv', index=False)
print('AG OOF:', predictor.leaderboard(silent=True).iloc[0]['score_val'])
print('saved submission_attack_4h.csv')

Verbosity: 2 (Standard Logging)


피쳐: 200개 | train: 30000 | test: 19995


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          4
Pytorch Version:    2.10.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.13 GB, Allocated: 0.00 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       29.38 GB / 31.35 GB (93.7%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 14400s
AutoGluon will save models to "/kaggle/working/ag_model"
Train Data Rows:    30000
Train Data Columns: 200
Label Column:       gender
Problem Type:       binary
Preprocessing data ...
Selected class <--> label ma

[1000]	valid_set's binary_logloss: 0.543873


	0.7204	 = Validation score   (roc_auc)
	113.81s	 = Training   runtime
	0.99s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 13075.20s of the 13075.19s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
No improvement since epoch 2: early stopping
No improvement since epoch 7: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 1: early stopping
No improvement since epoch 1: early stopping
No improvement since epoch 8: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 9: early stopping
	0.6918	 = Validation score   (roc_auc)
	357.92s	 = Training   runtime
	0.56s	 = Validation runtime
Fitting model: CatBoost_r9_BAG_L1 ... Training model for up to 12716.21s of the 12716.20s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequent

...

[1000]	valid_set's binary_logloss: 0.539462
[1000]	valid_set's binary_logloss: 0.543084
[1000]	valid_set's binary_logloss: 0.534429
[1000]	valid_set's binary_logloss: 0.545215
[2000]	valid_set's binary_logloss: 0.544928
[1000]	valid_set's binary_logloss: 0.546421
[1000]	valid_set's binary_logloss: 0.542444
[1000]	valid_set's binary_logloss: 0.542919
[2000]	valid_set's binary_logloss: 0.542212
[1000]	valid_set's binary_logloss: 0.546628
[2000]	valid_set's binary_logloss: 0.545663


	0.7238	 = Validation score   (roc_auc)
	371.27s	 = Training   runtime
	4.01s	 = Validation runtime
Fitting model: RandomForest_r39_BAG_L1 ... Training model for up to 5618.08s of the 5618.07s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=4, gpus=0, mem=0.1/26.7 GB
	0.701	 = Validation score   (roc_auc)
	526.83s	 = Training   runtime
	1.78s	 = Validation runtime
Fitting model: CatBoost_r167_BAG_L1 ... Training model for up to 5089.09s of the 5089.08s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	0.7188	 = Validation score   (roc_auc)
	128.2s	 = Training   runtime
	0.09s	 = Validation runtime
Fitting model: NeuralNetFastAI_r95_BAG_L1 ... Training model for up to 4960.46s of the 4960.45s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
No improvement since epoch 3: early

[1000]	valid_set's binary_logloss: 0.536741
[1000]	valid_set's binary_logloss: 0.542766
[1000]	valid_set's binary_logloss: 0.545437


	0.7199	 = Validation score   (roc_auc)
	68.14s	 = Training   runtime
	0.63s	 = Validation runtime
Fitting model: NeuralNetTorch_r158_BAG_L1 ... Training model for up to 3793.98s of the 3793.97s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	Ran out of time, stopping training early. (Stopping on epoch 111)
	0.7034	 = Validation score   (roc_auc)
	1931.31s	 = Training   runtime
	2.96s	 = Validation runtime
Fitting model: CatBoost_r86_BAG_L1 ... Training model for up to 1859.35s of the 1859.34s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	0.7213	 = Validation score   (roc_auc)
	440.65s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: NeuralNetFastAI_r37_BAG_L1 ... Training model for up to 1418.22s of the 1418.21s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with Sequenti

AG OOF: 0.7279895440845927
saved submission_attack_4h.csv


In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/icetraythegag/competition-data/feats_attack200.parquet
/kaggle/input/datasets/icetraythegag/competition-data/test_transactions.csv
/kaggle/input/datasets/icetraythegag/competition-data/y_train.csv
/kaggle/input/datasets/icetraythegag/competition-data/train_transactions.csv
